In [ ]:
import os
import random
import shutil
from google.colab import drive

# -------------------- Mount Google Drive --------------------
drive.mount('/content/drive')

# -------------------- Paths --------------------
base_dir = "/content/drive/MyDrive/tomato"  # folder containing disease folders
combined_base_dir = "/content/drive/MyDrive/tomato_combined"
os.makedirs(combined_base_dir, exist_ok=True)

image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
total_needed_per_disease = 500

# -------------------- Process each disease folder --------------------
for disease_folder in os.listdir(base_dir):
    disease_path = os.path.join(base_dir, disease_folder)
    if not os.path.isdir(disease_path):
        continue

    print(f"\nProcessing: {disease_folder}")

    # real folder inside disease folder
    real_dir = os.path.join(disease_path, "real")
    real_images = []
    if os.path.exists(real_dir):
        real_images = [os.path.join(real_dir, f) for f in os.listdir(real_dir)
                       if f.lower().endswith(image_exts)]

    # other images in disease folder (excluding 'real')
    main_images = [os.path.join(disease_path, f) for f in os.listdir(disease_path)
                   if f.lower().endswith(image_exts)]
    main_images = [img for img in main_images if img not in real_images]

    # Determine how many more images needed
    remaining_needed = total_needed_per_disease - len(real_images)
    if remaining_needed > 0:
        if len(main_images) < remaining_needed:
            print(f"⚠️ Only {len(real_images) + len(main_images)} images available in {disease_folder}")
            selected_main = main_images
        else:
            selected_main = random.sample(main_images, remaining_needed)
    else:
        selected_main = []

    # Final selection & shuffle
    selected_images = real_images + selected_main
    random.shuffle(selected_images)

    # Create output folder (same as disease folder name)
    output_folder = os.path.join(combined_base_dir, disease_folder)
    os.makedirs(output_folder, exist_ok=True)

    # Copy images
    for i, img_path in enumerate(selected_images, start=1):
        ext = os.path.splitext(img_path)[1]
        new_name = f"{disease_folder}_{i:03d}{ext}"
        shutil.copy(img_path, os.path.join(output_folder, new_name))

    print(f"✅ {len(selected_images)} images saved to {output_folder}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: Tomato__Tomato_YellowLeaf__Curl_Virus
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus

Processing: Tomato__Target_Spot
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato__Target_Spot

Processing: Tomato_Septoria_leaf_spot
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato_Septoria_leaf_spot

Processing: Tomato_Spider_mites_Two_spotted_spider_mite
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato_Spider_mites_Two_spotted_spider_mite

Processing: Tomato_Leaf_Mold
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato_Leaf_Mold

Processing: Tomato_Early_blight
✅ 500 images saved to /content/drive/MyDrive/tomato_combined/Tomato_Early_blight

Processing: Tomato__Tomato_mosaic_virus
✅ 500 images saved to /content/drive/MyDrive/tomato_comb

In [ ]:
print("\n🔎 Verifying combined images per disease folder:")

for disease_folder in os.listdir(combined_base_dir):
    folder_path = os.path.join(combined_base_dir, disease_folder)
    if not os.path.isdir(folder_path):
        continue
    # Count images in this combined folder
    images = [f for f in os.listdir(folder_path)
              if f.lower().endswith(image_exts)]
    print(f"{disease_folder}: {len(images)} images")



🔎 Verifying combined images per disease folder:
Tomato__Tomato_YellowLeaf__Curl_Virus: 500 images
Tomato__Target_Spot: 500 images
Tomato_Septoria_leaf_spot: 500 images
Tomato_Spider_mites_Two_spotted_spider_mite: 500 images
Tomato_Leaf_Mold: 500 images
Tomato_Early_blight: 500 images
Tomato__Tomato_mosaic_virus: 500 images
Tomato_Bacterial_spot: 500 images
Tomato_Late_blight: 500 images


In [ ]:
!pip install imagehash

import os
from PIL import Image, UnidentifiedImageError
import imagehash
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
data_dir = '/content/drive/MyDrive/tomato_combined'  # original dataset
clean_dir = '/content/drive/MyDrive/tomato_cleaned'  # cleaned images
os.makedirs(clean_dir, exist_ok=True)

# =======================
# 2️⃣ Remove duplicates & corrupt images & multi-leaf/text images
# =======================
hashes = {}

import os
from PIL import Image, UnidentifiedImageError
import imagehash

hashes = {}

for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(file_path)
            img.verify()  # check corrupt
            img = Image.open(file_path)  # reopen for hashing

            # Convert to RGB to handle RGBA / grayscale issues
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Compute perceptual hash
            h = imagehash.phash(img)
            if h in hashes:
                print(f"Duplicate skipped: {file_path}")
                continue
            else:
                hashes[h] = file_path

        except (UnidentifiedImageError, OSError, SyntaxError):
            print(f"Corrupt skipped: {file_path}")
            continue

        # Skip mostly empty images or text (grayscale extrema check)
        img_gray = img.convert("L")
        extrema = img_gray.getextrema()
        if extrema[1] - extrema[0] < 10:
            print(f"Empty/text image skipped: {file_path}")
            continue

        # Copy cleaned image to new folder with .jpg extension
        rel_path = os.path.relpath(file_path, data_dir)
        base_name = os.path.splitext(rel_path)[0] + ".jpg"  # force .jpg
        new_path = os.path.join(clean_dir, base_name)
        os.makedirs(os.path.dirname(new_path), exist_ok=True)

        # Save as JPEG
        img.save(new_path, format='JPEG')

print("✅ Dataset cleaning completed!")

# =======================
# 3️⃣ Aggressive Data Transform
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 4️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=clean_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 5️⃣ EfficientNet Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 6️⃣ Training Loop
# =======================
num_epochs = 15
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0,0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct/total
    avg_loss = running_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_cotton_model.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 7️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = transform_val(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.5 MB/s eta 0:00:00
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_056.png
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_058.jpg
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_106.png
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_122.jpg
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_136.jpg
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__Tomato_YellowLeaf__Curl_Virus/Tomato__Tomato_YellowLeaf__Curl_Virus_153.png
Duplicate skipped: /content/drive/MyDrive/tomato_combined/Tomato__To

100%|██████████| 20.5M/20.5M [00:00<00:00, 163MB/s]


Epoch [1/15] Loss: 1.2937 Val Acc: 0.6882
✅ Saved Best Model with Acc: 0.6882
Epoch [2/15] Loss: 0.8141 Val Acc: 0.7451
✅ Saved Best Model with Acc: 0.7451
Epoch [3/15] Loss: 0.6709 Val Acc: 0.7620
✅ Saved Best Model with Acc: 0.7620
Epoch [4/15] Loss: 0.5781 Val Acc: 0.7542
Epoch [5/15] Loss: 0.5020 Val Acc: 0.7555
Epoch [6/15] Loss: 0.4407 Val Acc: 0.7555
Epoch [7/15] Loss: 0.3826 Val Acc: 0.7658
✅ Saved Best Model with Acc: 0.7658
Epoch [8/15] Loss: 0.3404 Val Acc: 0.7723
✅ Saved Best Model with Acc: 0.7723
Epoch [9/15] Loss: 0.3251 Val Acc: 0.7723
Epoch [10/15] Loss: 0.3126 Val Acc: 0.7671
Epoch [11/15] Loss: 0.3111 Val Acc: 0.7749
✅ Saved Best Model with Acc: 0.7749
Epoch [12/15] Loss: 0.3190 Val Acc: 0.7775
✅ Saved Best Model with Acc: 0.7775
Epoch [13/15] Loss: 0.3029 Val Acc: 0.7736
Epoch [14/15] Loss: 0.3135 Val Acc: 0.7658
Epoch [15/15] Loss: 0.3254 Val Acc: 0.7749


In [ ]:
drive_model_path = "/content/drive/MyDrive/best_tomato_model.pth"
torch.save(model.state_dict(), drive_model_path)
print(f"✅ Model weights saved to: {drive_model_path}")


✅ Model weights saved to: /content/drive/MyDrive/best_tomato_model.pth
